# Phase C — 5 Transformer Fine-tuning (multi-label)

5 pretrained transformer (**ViT · DeiT · BeiT · Swin · CvT**) afişten tür tahmini için fine-tune edilir. 5-fold CV → 25 koşu. **Colab/GPU içindir.**

**Akış:** önce **pilot** (1 model × 1 fold) ile T4'te süreyi ölç + pipeline doğrula → iyiyse `RUN_FULL=True` ile tam 25 koşu.

**Çıktı (Drive):** `checkpoints/<model>_fold<f>_best.pt`, `oof/<model>_fold<f>.npz` (Phase D için OOF tahminler), `.json` (loss geçmişi + train/inference time).

## 0) Kurulum

In [ ]:
import os, sys, time, json, zipfile, random
from pathlib import Path

# Colab:
# !pip -q install transformers accelerate

import numpy as np
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Yerel ---
DATA_ROOT = Path('../')
# --- Colab (yukaridakini yorumlayip bunlari ac) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = Path('/content/drive/MyDrive/film-genre-project-data')

CKPT_ROOT = DATA_ROOT / 'checkpoints'; CKPT_ROOT.mkdir(parents=True, exist_ok=True)
OOF_DIR   = DATA_ROOT / 'oof';         OOF_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT.resolve())

## 1) Veriyi hazırla (zip'i Drive'da bir kez aç)

In [ ]:
# film-genre-data-v2.zip'i Drive'daki DATA_ROOT'a yukledikten sonra:
zip_path = DATA_ROOT / 'film-genre-data-v2.zip'
if not (DATA_ROOT / 'posters').exists() and zip_path.exists():
    print('Zip aciliyor (bir kez)...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_ROOT)
    print('Acildi.')

assert (DATA_ROOT / 'labels_v2.csv').exists() and (DATA_ROOT / 'folds').exists(), "Veri yok - zip Drive icine yuklenip acildi mi?"
n_post = sum(1 for _ in (DATA_ROOT / 'posters').glob('*.jpg'))
print('Veri hazir | poster:', n_post)

## 2) Config

In [ ]:
MODELS = {
    'vit':  'google/vit-base-patch16-224',
    'deit': 'facebook/deit-base-distilled-patch16-224',
    'beit': 'microsoft/beit-base-patch16-224-pt22k-ft22k',
    'swin': 'microsoft/swin-base-patch4-window7-224',
    'cvt':  'microsoft/cvt-13',
}
N_SPLITS     = 5
BATCH        = 32        # T4 16GB icin guvenli; OOM olursa 16 yap
EPOCHS       = 6
LR           = 5e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS  = 2
SEED         = 42

# Pilot: once tek model x tek fold (T4 suresini olc)
PILOT_MODEL = 'vit'
PILOT_FOLD  = 0

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

## 3) Dataset + transforms

In [ ]:
import pickle
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

with open(DATA_ROOT / 'mlb.pkl', 'rb') as f:
    mlb = pickle.load(f)
CLASSES = list(mlb.classes_); N_CLASSES = len(CLASSES)
POSTERS = DATA_ROOT / 'posters'
print('Siniflar (%d):' % N_CLASSES, CLASSES)

class PosterDataset(Dataset):
    def __init__(self, df, tfm):
        self.ids = df['tmdb_id'].astype(str).tolist()
        self.labels = mlb.transform(df['genres'].apply(lambda x: x.split('|'))).astype('float32')
        self.tfm = tfm
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        img = Image.open(POSTERS / (self.ids[i] + '.jpg')).convert('RGB')
        return self.tfm(img), torch.from_numpy(self.labels[i])

# Posterler 2:3 portre -> kareye (size x size) resize (distort). Karari rapora not.
def build_transforms(mean, std, size):
    train = T.Compose([T.Resize((size, size)), T.ColorJitter(0.1, 0.1, 0.1),
                       T.ToTensor(), T.Normalize(mean, std)])
    val   = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(mean, std)])
    return train, val

## 4) Model factory + metrik

In [ ]:
from transformers import (AutoModelForImageClassification, AutoImageProcessor,
                          get_cosine_schedule_with_warmup)
from sklearn.metrics import f1_score

def build_model(hf_id):
    return AutoModelForImageClassification.from_pretrained(
        hf_id, num_labels=N_CLASSES,
        problem_type='multi_label_classification',
        ignore_mismatched_sizes=True).to(device)

def processor_stats(hf_id):
    p = AutoImageProcessor.from_pretrained(hf_id)
    mean = list(getattr(p, 'image_mean', [0.485, 0.456, 0.406]))
    std  = list(getattr(p, 'image_std',  [0.229, 0.224, 0.225]))
    size = 224
    s = getattr(p, 'size', None)
    if isinstance(s, dict):
        size = s.get('height') or s.get('shortest_edge') or 224
    return mean, std, int(size)

def macro_f1(probs, targets, thr=0.5):
    return f1_score(targets, (probs >= thr).astype(int), average='macro', zero_division=0)

## 5) Tek fold eğit + değerlendir

In [ ]:
def train_one_fold(model_key, fold, epochs=EPOCHS):
    hf_id = MODELS[model_key]
    mean, std, size = processor_stats(hf_id)
    tfm_tr, tfm_va = build_transforms(mean, std, size)

    tr = pd.read_csv(DATA_ROOT / 'folds' / ('fold_%d_train.csv' % fold), dtype={'tmdb_id': str})
    va = pd.read_csv(DATA_ROOT / 'folds' / ('fold_%d_val.csv'   % fold), dtype={'tmdb_id': str})
    dl_tr = DataLoader(PosterDataset(tr, tfm_tr), batch_size=BATCH, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True)
    dl_va = DataLoader(PosterDataset(va, tfm_va), batch_size=BATCH, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model(hf_id)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = len(dl_tr) * epochs
    sch = get_cosine_schedule_with_warmup(opt, int(0.1 * steps), steps)
    lossfn = torch.nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler()

    hist = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    best_f1, best_state, best_probs, val_targets = -1.0, None, None, None
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tl = 0.0
        for x, y in dl_tr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(pixel_values=x).logits
                loss = lossfn(logits, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
            tl += loss.item() * len(x)
        tl /= len(dl_tr.dataset)

        model.eval(); vl = 0.0; P = []; Yt = []
        with torch.no_grad():
            for x, y in dl_va:
                x = x.to(device)
                with torch.cuda.amp.autocast():
                    logits = model(pixel_values=x).logits
                    loss = lossfn(logits, y.to(device))
                vl += loss.item() * len(x)
                P.append(torch.sigmoid(logits).float().cpu().numpy()); Yt.append(y.numpy())
        vl /= len(dl_va.dataset)
        P = np.concatenate(P); Yt = np.concatenate(Yt)
        f1 = macro_f1(P, Yt)
        hist['train_loss'].append(tl); hist['val_loss'].append(vl); hist['val_f1'].append(f1)
        print('  [%s f%d] ep %d/%d  train=%.4f val=%.4f macroF1=%.4f'
              % (model_key, fold, ep + 1, epochs, tl, vl, f1))
        if f1 > best_f1:
            best_f1, best_probs, val_targets = f1, P, Yt
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    train_time = time.time() - t0

    # inference time (ms/img)
    model.eval(); n = 0; t1 = time.time()
    with torch.no_grad():
        for x, y in dl_va:
            x = x.to(device)
            with torch.cuda.amp.autocast():
                _ = model(pixel_values=x).logits
            n += len(x)
    infer_ms = (time.time() - t1) / max(n, 1) * 1000

    tag = '%s_fold%d' % (model_key, fold)
    torch.save(best_state, CKPT_ROOT / (tag + '_best.pt'))
    np.savez(OOF_DIR / (tag + '.npz'), probs=best_probs, targets=val_targets,
             ids=va['tmdb_id'].values, classes=np.array(CLASSES))
    json.dump({'hist': hist, 'best_f1': best_f1, 'train_time_s': train_time,
               'infer_ms_per_img': infer_ms}, open(OOF_DIR / (tag + '.json'), 'w'))
    print('  -> best macroF1=%.4f | train=%.1fs | infer=%.2f ms/img' % (best_f1, train_time, infer_ms))
    return hist, best_f1, train_time, infer_ms

## 6) PILOT — 1 model × 1 fold (T4 süresini ölç)

In [ ]:
import matplotlib.pyplot as plt

print('=== PILOT: %s x fold %d ===' % (PILOT_MODEL, PILOT_FOLD))
hist, bf1, tt, it = train_one_fold(PILOT_MODEL, PILOT_FOLD)

ep = range(1, len(hist['train_loss']) + 1)
plt.figure(figsize=(7, 4))
plt.plot(ep, hist['train_loss'], marker='o', label='train')
plt.plot(ep, hist['val_loss'],   marker='o', label='val')
plt.xlabel('epoch'); plt.ylabel('BCE loss'); plt.legend()
plt.title('%s fold%d - loss' % (PILOT_MODEL, PILOT_FOLD)); plt.show()

est_min = tt * len(MODELS) * N_SPLITS / 60
print('Pilot train suresi: %.1f dk' % (tt / 60))
print('Tahmini TAM 25 kosu (ayni hizda): %.0f dk = %.1f saat' % (est_min, est_min / 60))

## 7) Tam 25 koşu (pilot iyiyse `RUN_FULL=True`)

In [ ]:
RUN_FULL = False  # pilot sonucu iyiyse True yapip calistir

if RUN_FULL:
    summary = {}
    for mk in MODELS:
        for fold in range(N_SPLITS):
            print('=== %s fold %d ===' % (mk, fold))
            h, f1, tt, it = train_one_fold(mk, fold)
            summary['%s_fold%d' % (mk, fold)] = {'best_f1': f1, 'train_s': tt, 'infer_ms': it}
    json.dump(summary, open(OOF_DIR / 'summary.json', 'w'), indent=2)
    print('TAM kosu bitti. OOF + checkpointler Drive icinde.')
else:
    print('RUN_FULL=False. Pilot iyiyse bu hucrede RUN_FULL=True yapip tekrar calistir.')